# Phase 1 — Benchmark chung
ARC-Easy và HellaSwag qua `lm-evaluation-harness` (PLAN 3.3): 500 câu đầu mỗi bộ, 0-shot, chấm theo log-likelihood. Không cần token HF. GPU nhanh hơn; trên CPU đặt `DEVICE = "cpu"`.


## 1. Cài thư viện

In [ ]:
%pip install -q transformers==5.20.2 huggingface-hub==1.32.0 psutil==7.2.2 safetensors==0.8.0 pyyaml==6.0.3 datasets==5.1.0 lm_eval==0.4.13
from importlib.metadata import version
wanted = {"transformers": "5.20.2", "huggingface-hub": "1.32.0", "psutil": "7.2.2", "safetensors": "0.8.0", "pyyaml": "6.0.3", "datasets": "5.1.0", "lm_eval": "0.4.13"}
installed = {name: version(name) for name in wanted}
print(installed)
assert installed == wanted, "Phiên bản chưa khớp requirements.txt; chạy lại cell này."


## 2. Nạp mã nguồn của dự án

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "1169ad330e86f00718836b46eb6e791e16be4d7d84d08fea2259795ca07731c8"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "1169ad330e86f00718836b46eb6e791e16be4d7d84d08fea2259795ca07731c8")
print("Working directory:", root)


## 3. Kiểm tra mã

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr)
checks.check_returncode()


## 4. Chạy ARC-Easy và HellaSwag
`MODEL` là khóa trong `configs/models.yaml` hoặc id trên Hub. Kết quả ghi vào `results/benchmarks/`.

In [ ]:
import torch
MODEL = "chat_reference"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
!python -m src.eval.benchmarks --model {MODEL} --device {DEVICE} --batch-size 16
